In [48]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import LabelEncoder

from sklearn.feature_extraction.text import CountVectorizer
import joblib


In [49]:
df = pd.read_csv("/Users/alexkochu/Desktop/Career_Architect/student_career_data.csv")

In [50]:
df.head()

,Student_ID,CGPA,Interests,Skills,Courses_Completed,Projects_Count,Role
0,bddfa350-14e3-4df4-acc7-b53c56b17991,8.66,Artificial Intelligence,"Scikit-learn,PyTorch,NoSQL,Angular",17,2,AI/ML Engineer
1,2318ad95-567f-4a1e-970b-b74c07352472,8.82,Artificial Intelligence,"PyTorch,Scikit-learn,Python,NumPy,C++",7,4,AI/ML Engineer
2,c6949319-e7a2-4803-98c9-7a026f8cd5e6,9.62,Artificial Intelligence,"Python,Mathematics,PyTorch,Scikit-learn,Angula...",14,1,AI/ML Engineer
3,9777b848-eaeb-4fc6-aaf3-26ef842c3b07,7.02,Machine Learning,"Excel,Tableau,Pandas,SQL,Azure,C++,PyTorch",20,3,Data Analyst
4,d0b244aa-bd1f-4dfa-afce-946209ed3f4a,8.15,Database Management,"Pandas,Tableau,Python,Angular",19,5,Data Analyst


In [51]:
df.isnull().sum()

Student_ID           0
CGPA                 0
Interests            0
Skills               0
Courses_Completed    0
Projects_Count       0
Role                 0
dtype: int64

In [52]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Student_ID         2000 non-null   object 
 1   CGPA               2000 non-null   float64
 2   Interests          2000 non-null   object 
 3   Skills             2000 non-null   object 
 4   Courses_Completed  2000 non-null   int64  
 5   Projects_Count     2000 non-null   int64  
 6   Role               2000 non-null   object 
dtypes: float64(1), int64(2), object(4)
memory usage: 109.5+ KB


In [53]:
df = df.drop(columns=["Student_ID"])

In [54]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CGPA               2000 non-null   float64
 1   Interests          2000 non-null   object 
 2   Skills             2000 non-null   object 
 3   Courses_Completed  2000 non-null   int64  
 4   Projects_Count     2000 non-null   int64  
 5   Role               2000 non-null   object 
dtypes: float64(1), int64(2), object(3)
memory usage: 93.9+ KB


In [55]:
le_interest = LabelEncoder()
le_role = LabelEncoder()

df["Interests"] = le_interest.fit_transform(df["Interests"])
df["Role"] = le_role.fit_transform(df["Role"])

In [56]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CGPA               2000 non-null   float64
 1   Interests          2000 non-null   int64  
 2   Skills             2000 non-null   object 
 3   Courses_Completed  2000 non-null   int64  
 4   Projects_Count     2000 non-null   int64  
 5   Role               2000 non-null   int64  
dtypes: float64(1), int64(4), object(1)
memory usage: 93.9+ KB


In [57]:
df.head()

,CGPA,Interests,Skills,Courses_Completed,Projects_Count,Role
0,8.66,1,"Scikit-learn,PyTorch,NoSQL,Angular",17,2,0
1,8.82,1,"PyTorch,Scikit-learn,Python,NumPy,C++",7,4,0
2,9.62,1,"Python,Mathematics,PyTorch,Scikit-learn,Angula...",14,1,0
3,7.02,7,"Excel,Tableau,Pandas,SQL,Azure,C++,PyTorch",20,3,2
4,8.15,5,"Pandas,Tableau,Python,Angular",19,5,2


In [58]:
def skill_tokenizer(text):
    return text.split(",")

vectorizer = CountVectorizer(tokenizer=skill_tokenizer)

skills_matrix = vectorizer.fit_transform(df["Skills"])
skills_df = pd.DataFrame(skills_matrix.toarray(),
                         columns=vectorizer.get_feature_names_out())

/opt/anaconda3/lib/python3.13/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [59]:
X_numeric = df[["CGPA", "Interests", "Courses_Completed", "Projects_Count"]]
X = pd.concat([X_numeric, skills_df], axis=1)

y = df["Role"]

In [60]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [61]:
rf = RandomForestClassifier(n_estimators=300, random_state=42)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Accuracy: 0.94

Classification Report:

              precision    recall  f1-score   support

           0       0.98      1.00      0.99        47
           1       0.99      0.99      0.99        72
           2       0.89      0.88      0.88        80
           3       0.80      0.83      0.81        58
           4       1.00      0.96      0.98        67
           5       0.99      1.00      0.99        76

    accuracy                           0.94       400
   macro avg       0.94      0.94      0.94       400
weighted avg       0.94      0.94      0.94       400



In [62]:
kmeans = KMeans(n_clusters=5, random_state=42)
clusters = kmeans.fit_predict(X)

df["Cluster"] = clusters
print(df.head())

   CGPA  Interests                                             Skills  \
0  8.66          1                 Scikit-learn,PyTorch,NoSQL,Angular   
1  8.82          1              PyTorch,Scikit-learn,Python,NumPy,C++   
2  9.62          1  Python,Mathematics,PyTorch,Scikit-learn,Angula...   
3  7.02          7         Excel,Tableau,Pandas,SQL,Azure,C++,PyTorch   
4  8.15          5                      Pandas,Tableau,Python,Angular   

   Courses_Completed  Projects_Count  Role  Cluster  
0                 17               2     0        0  
1                  7               4     0        2  
2                 14               1     0        0  
3                 20               3     2        0  
4                 19               5     2        0  


In [63]:
joblib.dump(rf, "career_rf_model.pkl")
joblib.dump(kmeans, "career_cluster_model.pkl")
joblib.dump(vectorizer, "skill_vectorizer.pkl")
joblib.dump(le_interest, "interest_encoder.pkl")
joblib.dump(le_role, "role_encoder.pkl")

['role_encoder.pkl']